In [2]:
import pandas as pd
import numpy as np
import re
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

tamil = pd.read_csv("tamil.txt", sep="\t", encoding="latin1")
english = pd.read_csv("english.txt", sep="\t", encoding="latin1")
tanglish = pd.read_csv("tanglish.txt", sep="\t", encoding="latin1")


In [3]:

df = pd.concat([tamil, english, tanglish], ignore_index=True)

print("Total rows:", len(df))
df.head()
df = df.iloc[:, :2]
df.columns = ["comment", "label"]

df.drop_duplicates(subset="comment", inplace=True)
df.dropna(inplace=True)
df = df[df["comment"].str.strip() != ""]

print("After cleanup:", len(df))

def clean_text(text):
    text = str(text).lower()

    # Remove URLs & mentions only
    text = re.sub(r"http\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)

    # Normalize obfuscated profanity
    text = re.sub(r"b[\W_]*i[\W_]*t[\W_]*c[\W_]*h", " bitch ", text)
    text = re.sub(r"s[\W_]*h[\W_]*i[\W_]*t", " shit ", text)

    # Keep EVERYTHING else (important)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


Total rows: 35039
After cleanup: 38


In [4]:

df["comment"] = df["comment"].apply(clean_text)
print("After cleanup:", len(df))

df.head()

le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])

for label, code in zip(le.classes_, le.transform(le.classes_)):
    print(label, "→", code)

X = df["comment"]
y = df["label_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", len(X_train))
print("Test:", len(X_test))

tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    min_df=3,
    sublinear_tf=True
)


After cleanup: 38
non_toxic → 0
severe_toxic → 1
toxic → 2
Train: 30
Test: 8


In [5]:
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print(X_train_vec.shape)

model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)


model.fit(X_train_vec, y_train)
print("Model trained")

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

STRONG_TOXIC_WORDS = {
    # English
    "bitch", "die", "kill", "stupid", "idiot",

    # Tanglish
    "sethudu", "sethudanum", "saavu", "poi sethudu",
    "waste", "loosu", "mokka",

    # Tamil unicode (partial examples)
    "சாக", "செத்து", "கொல்ல"
}

SEVERE_TOXIC_WORDS = {
    "kill", "die", "sethudu", "sethudanum", "poi sethudu",
    "saavu", "kollanum", "செத்து", "கொல்ல"
}

MILD_TOXIC_WORDS = {
    "bitch", "idiot", "stupid", "loosu", "waste", "mokka"
}



(30, 12)
Model trained
Accuracy: 0.375
              precision    recall  f1-score   support

   non_toxic       0.33      0.50      0.40         2
severe_toxic       0.00      0.00      0.00         3
       toxic       0.50      0.67      0.57         3

    accuracy                           0.38         8
   macro avg       0.28      0.39      0.32         8
weighted avg       0.27      0.38      0.31         8

Accuracy: 0.375
              precision    recall  f1-score   support

   non_toxic       0.33      0.50      0.40         2
severe_toxic       0.00      0.00      0.00         3
       toxic       0.50      0.67      0.57         3

    accuracy                           0.38         8
   macro avg       0.28      0.39      0.32         8
weighted avg       0.27      0.38      0.31         8



In [6]:

def contains_severe_toxic(text):
    return any(w in text for w in SEVERE_TOXIC_WORDS)

def contains_mild_toxic(text):
    return any(w in text for w in MILD_TOXIC_WORDS)


def contains_strong_toxic(text):
    tokens = text.split()
    for w in tokens:
        if w in STRONG_TOXIC_WORDS:
            return True
    return False

def predict_comment(text):
    cleaned = clean_text(text)

    # 🚨 HARD BLOCK: SEVERE
    if contains_severe_toxic(cleaned):
        return "severe_toxic"

    vec = tfidf.transform([cleaned])
    probs = model.predict_proba(vec)[0]

    idx_non = le.transform(["non_toxic"])[0]
    idx_tox = le.transform(["toxic"])[0]
    idx_sev = le.transform(["severe_toxic"])[0]

    # Mild toxic override only if ML agrees
    if contains_mild_toxic(cleaned):
        if probs[idx_tox] + probs[idx_sev] > 0.20:
            return "toxic"

    return le.inverse_transform([np.argmax(probs)])[0]

In [7]:

tests = [
    "you good",
    "nice explanation bitch",
    "super video idiot",
    "well done",
    "poi sethudu da",
    "kill yourself",
    "very useful content"
]

for t in tests:
    print(t, "→", predict_comment(t))


you good → non_toxic
nice explanation bitch → toxic
super video idiot → toxic
well done → non_toxic
poi sethudu da → severe_toxic
kill yourself → severe_toxic
very useful content → non_toxic
